In [1]:
import json_tricks
answer = {}


inputs = json_tricks.load(open('inputs/inputs.json', 'r'))

# Backpropagation Practice

You are given a graph of operations:

<img src="imgs/task_2_4_graph_02.png" width="400" height="300" />

# Task 1

Write a function that calculates the value of this graph for any given input vector `x` and set of coefficients $b_1, b_2, c_1, c_2$ packed as vector of weights `w`.
In this function also return all the $z$ values.

In [ ]:
import numpy as np

def graph_value(x, w):
    x1, x2 = x[0], x[1]
    b1, b2, c1, c2 = w[0], w[1], w[2], w[3]

    z1 = x1 + b1
    z2 = x2 + b2
    z3 = 1 / (1 + np.exp(-z1))
    z4 = 1 / (1 + np.exp(-z2))
    z5 = np.tanh(z2)
    z6 = z5 * c2
    z7 = z1 * z4
    z8 = z7 * c1
    z9 = z3 * z6

    y = z8 + z9
    z = [z1, z2, z3, z4, z5, z6, z7, z8, z9]
    return y, z

In [3]:
answer['graph_value'] = [graph_value(**input) for input in inputs]

# Task 2

In this graph, find all direct derivatives of all the values over $z$ or $x$ or $w$.

For example, if $y = z_8 + z_9$, you need to find only $\partial_{z_9} y$ and $\partial_{z_8} y$ for that operation.

You have to do that for all the intermediate signals.

Write the result in the form of a dictionary, for example, $\partial_{z_9} y$:

```
['dy_dz9'] = 1
```

In [ ]:
def graph_derivatives(x, w):
    y, z = graph_value(x, w)
    z1, z2, z3, z4, z5, z6, z7, z8, z9 = z
    b1, b2, c1, c2 = w[0], w[1], w[2], w[3]
    res = {}

    res['dz1_dx1'] = 1.0
    res['dz1_db1'] = 1.0
    res['dz2_dx2'] = 1.0
    res['dz2_db2'] = 1.0

    res['dz3_dz1'] = z3 * (1 - z3)
    res['dz4_dz2'] = z4 * (1 - z4)
    res['dz5_dz2'] = 1 - z5 ** 2

    res['dz6_dz5'] = c2
    res['dz6_dc2'] = z5

    res['dz7_dz1'] = z4
    res['dz7_dz4'] = z1

    res['dz8_dz7'] = c1
    res['dz8_dc1'] = z7

    res['dz9_dz3'] = z6
    res['dz9_dz6'] = z3

    res['dy_dz8'] = 1.0
    res['dy_dz9'] = 1.0

    return res

In [5]:
answer['graph_derivatives'] = [graph_derivatives(**input) for input in inputs]

# Task 3

Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_1} y$
- calculate that derivative

In [ ]:
def dy_dc1(x, w):
    derivs = graph_derivatives(x, w)
    selected_derivs = {
        'dy_dz8': derivs['dy_dz8'],
        'dz8_dc1': derivs['dz8_dc1'],
    }
    dy_dc1 = selected_derivs['dy_dz8'] * selected_derivs['dz8_dc1']
    return selected_derivs, dy_dc1

In [7]:
answer['dy_dc1'] = [dy_dc1(**input) for input in inputs]

# Task 4
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_2} y$
- calculate that derivative

In [ ]:
def dy_dc2(x, w):
    derivs = graph_derivatives(x, w)
    selected_derivs = {
        'dy_dz9': derivs['dy_dz9'],
        'dz9_dz6': derivs['dz9_dz6'],
        'dz6_dc2': derivs['dz6_dc2'],
    }
    dy_dc2 = selected_derivs['dy_dz9'] * selected_derivs['dz9_dz6'] * selected_derivs['dz6_dc2']
    return selected_derivs, dy_dc2

In [ ]:
answer['dy_dc2'] = [dy_dc2(**input) for input in inputs]

# Task 5
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_1} y$
- calculate that derivative

In [ ]:
def dy_db1(x, w):
    derivs = graph_derivatives(x, w)
    selected_derivs = {
        'dy_dz8': derivs['dy_dz8'], 'dz8_dz7': derivs['dz8_dz7'], 'dz7_dz1': derivs['dz7_dz1'],
        'dy_dz9': derivs['dy_dz9'], 'dz9_dz3': derivs['dz9_dz3'], 'dz3_dz1': derivs['dz3_dz1'],
        'dz1_db1': derivs['dz1_db1'],
    }
    path_via_z7 = selected_derivs['dy_dz8'] * selected_derivs['dz8_dz7'] * selected_derivs['dz7_dz1']
    path_via_z3 = selected_derivs['dy_dz9'] * selected_derivs['dz9_dz3'] * selected_derivs['dz3_dz1']
    dy_db1 = selected_derivs['dz1_db1'] * (path_via_z7 + path_via_z3)
    return selected_derivs, dy_db1

In [11]:
answer['dy_db1'] = [dy_db1(**input) for input in inputs]

# Task 6
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_2} y$
- calculate that derivative

In [ ]:
def dy_db2(x, w):
    derivs = graph_derivatives(x, w)
    selected_derivs = {
        'dy_dz8': derivs['dy_dz8'], 'dz8_dz7': derivs['dz8_dz7'], 'dz7_dz4': derivs['dz7_dz4'], 'dz4_dz2': derivs['dz4_dz2'],
        'dy_dz9': derivs['dy_dz9'], 'dz9_dz6': derivs['dz9_dz6'], 'dz6_dz5': derivs['dz6_dz5'], 'dz5_dz2': derivs['dz5_dz2'],
        'dz2_db2': derivs['dz2_db2'],
    }
    path_via_z4 = selected_derivs['dy_dz8'] * selected_derivs['dz8_dz7'] * selected_derivs['dz7_dz4'] * selected_derivs['dz4_dz2']
    path_via_z5 = selected_derivs['dy_dz9'] * selected_derivs['dz9_dz6'] * selected_derivs['dz6_dz5'] * selected_derivs['dz5_dz2']
    dy_db2 = selected_derivs['dz2_db2'] * (path_via_z4 + path_via_z5)
    return selected_derivs, dy_db2

In [13]:
answer['dy_db2'] = [dy_db2(**input) for input in inputs]

In [14]:
json_tricks.dump(answer, '.answer.json')